<a href="https://colab.research.google.com/github/Muffin-Man2497/VarroAI/blob/main/bees.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Data
Schurischuster Stefan, & Martin Kampel. (2020). VarroaDataset [Data set]. Zenodo. http://doi.org/10.5281/zenodo.4085043

Download small versions or full sets:
https://drive.google.com/drive/folders/1KrnsAYyG3bVDk1qSRUKeAxh_-09ub-XV?usp=drive_link

I found it best to move the downloads into sample_data for easier setup and move them out when you extract them.

Make sure that when you extract, you extract to a directory so nothing gets flooded with massive amounts (>8,000) of images or txt files.

**IMPORTANT** Each of the zip files should have a "labels" directory and an "images" directory, although some may have an error where it was "labels" and "videos" instead. Either way provide the image directory when giving input to the model in the next sections.

In [ ]:
!rm -rf /content/full/

In [1]:
!cp /content/drive/MyDrive/VarroAI/Training\ Data/full/Full.zip /content/sample_data/

In [4]:
#The Full zip has a directory already containing both the labels and image folders with their portions
#!rm -rf /content/full/ # remove any already existing dir
!cp /content/drive/MyDrive/VarroAI/Training\ Data/full/Full.zip /content/sample_data/ # transfer from drive (specific to my setup)
!unzip /content/sample_data/Full.zip -d /content/

mv: cannot stat '/content/drive/MyDrive/VarroAI/Training Data/full/Full.zip': No such file or directory
unzip:  cannot find or open /content/sample_data/Full.zip, /content/sample_data/Full.zip.zip or /content/sample_data/Full.zip.ZIP.


##Dataloader Class

A custom dataloader class requires a `__len__` and `__getitem__` method, which helps standardize it so it can be loaded with some functionality. The below transform is a series of conversion of a PIL to an image, and then to a tensor of torch.float32 datatype, and then normalized. The final panel is the entire experiment wrapped into a neat dataloader class that shows the transform for the data, and loading it into the class. The `__getlabel__` method was particularly difficult and required some string slicing that is only featured on the last panel because it was dependent on the other method, `__getitem__`

In [ ]:
import torchvision
from torchvision.transforms import v2
from PIL import Image
import torch.utils.data
#Defining and testing the transform

img = Image.open("/content/sample_data/Test.png").convert("RGB") #opens image, ensures rgb mode

transform = v2.Compose([v2.ToImage(), #Converts PIL, tensors and ndarrays into TV (torch vision) Images
                 v2.ToDtype(torch.float32, scale=True), #converts the image to torch.float32 tensor
                 v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]) #normalizes with mean of 0th arg and std of 1st arg

print(transform(img))

FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_data/Test.png'

In [ ]:
from pathlib import Path
import torch.utils.data

transform = v2.Compose([v2.ToImage(), #Converts PIL, tensors and ndarrays into TV Images
                 v2.ToDtype(torch.float32, scale=True), #converts the image to torch.float32 tensor
                 v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]) #normalizes with mean of 0th arg and std of 1st arg


#-----------=============== Post-Definition Code ===============-----------#

class Custom(torch.utils.data.Dataset):
  def __init__(self, root_dir):
    self.root_dir = Path(root_dir)
  def __len__(self):
    return 1 #len
  def __getitem__(self, idx):
    img = Image.open(self.root_dir)
    img = transform(img)
    return img[idx]

customset = Custom("/content/sample_data/Test.png")
dataloader = torch.utils.data.DataLoader(customset, batch_size=1, shuffle=True) #transform defined below so i removed it from the top panel

idx = int(input("index: "))
print(customset[idx]) # c h w

index: 0
tensor([[-0.3098, -0.3020, -0.3255,  ..., -0.3176, -0.3255, -0.3333],
        [-0.2784, -0.2784, -0.3020,  ..., -0.3020, -0.3176, -0.3333],
        [-0.2706, -0.2941, -0.3098,  ..., -0.2941, -0.3176, -0.3255],
        ...,
        [-0.2314, -0.1608, -0.1294,  ..., -0.1608, -0.1529, -0.1294],
        [-0.1843, -0.1216, -0.1216,  ..., -0.1765, -0.1529, -0.1137],
        [-0.2000, -0.2000, -0.2471,  ..., -0.1686, -0.1608, -0.1294]])


In [3]:
import torch
from PIL import Image
from pathlib import Path
import torch.utils.data
import torchvision
from torchvision.transforms import v2
import os

transforms= v2.Compose([v2.ToImage(), #Converts PIL, tensors and ndarrays into TV Images
                        v2.Resize([160,280]), #resizes to a standard for CNN
                 v2.ToDtype(torch.float32, scale=True), #converts the image to torch.float32 tensor
                 v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]) #normalizes with mean of 0th arg and std of 1st arg

class Custom(torch.utils.data.Dataset):
  def __init__(self, image_dir, transform):
    self.label_dir = image_dir[:str(image_dir).rfind("/")]
    self.image_dir = Path(image_dir)
    self.transform=transform
    self.images = [
        os.path.join(image_dir, iname) for iname in os.listdir(image_dir)
    ] ####
  def __len__(self):
    return len(self.images) #len
  def __getlabel__(self,id):
    label = self.images[id] #grab image from file system
    label = label[label.rfind("/")+1:-4] #get only the name of the file and exclude extensions
    label = os.path.join(self.label_dir,"labels", label+".txt") #get label from labels/name.txt
    try:
      with open(label) as fileInput:
        label = fileInput.readline().strip()
      if label == "0":
        return 0
      else:
        return 1
    except FileNotFoundError:
      pass
  def __getitem__(self, id):
    if ".DS_Store" in self.images[id]: #File that continued to pop into my directory; necessary on google ?
      pass
    else:
      image = Image.open(self.images[id]).convert('RGB')
      img = self.transform(image)
      label = int(self.__getlabel__(id)) #ensure correct datatype
      return torch.tensor(img)

file_input = input("Path: ")
customset = Custom(file_input, transform=transforms)
dataloader = torch.utils.data.DataLoader(customset, batch_size=customset.__len__(), shuffle=True)
datax = []
datay = []
for i in range(customset.__len__()):
  if customset.__getlabel__(i) is not None: # accounting for image files without labels
    datax.append(customset[i])
    datay.append(int(customset.__getlabel__(i)))
datax = torch.stack(datax)
print(datax.shape)
datay = torch.tensor(datay)
print(datay.shape)

Path: /content/Full/images


/tmp/ipykernel_756/2148555953.py:44: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(img)


torch.Size([8218, 3, 160, 280])
torch.Size([8218])


##The Hard Part (custom loss, check out at the end if you like)

In hindsight, this loss function did not work, but I thought the thought process could be useful. Over all, it simply wasn't effective in my use case likely because I misunderstood its required input when using it sequentially or because of poor hyperparameters, although I spent 2 hours trying to fine tune it. It seems that the loss remained really small and the model just didn't seem to need to adjust its weights, maybe similar to the vanishing gradient problem.

For most classification problems there is going to be some bias. As commonly demonstrated by, credit card fraud could be more than 999 to 1, which would make the model 99.9% accurate if it solely guessed it wasn't fraud. The best ways to make it more accurate for that 0.1% is to either over-sample, or apply more heavy loss if it incorrectly classifies the minority class. Other ways include methods like under sampling, except that would be ineffective for this problem. I made the decision to use dynamic loss, where incorrectly classifying the minority class would cause more loss. This is because over sampling in images could require Synthetic Minority Over Sampling Technique (SMOTE), which is difficult to implement in a CNN due to the timing of application. Anyway, this loss will be implemented using Focal Loss, which was introduced in this [paper](https://arxiv.org/pdf/1708.02002) and is essentially BCE with a scaling factor in front of it.

-(1-p<sub>t</sub>)<sup>$\gamma$</sup> * log(p<sub>t</sub>)

To break this down, p<sub>t</sub> is how certain the models prediction of the correct class was. If it guessed 70% the right class p<sub>t</sub> would be 0.7, and if it guessed 20%, p<sub>t</sub> would be 0.2. The gamma parameter is representative of how aggressively easy examples are suppressed. For example if the value is really really high, like a 5, the end result is going to be very low, because anything between 0 and 1 multiplied by itself numerous times is going to be less than it originally was, and the value of the (1-p<sub>t</sub>) is always going to be less than one (hopefully). The typical starting value is around 2. Now that we have defined the function and understood how it works, an example of why it's effective is below.

**Keep in mind that log(0) errors can occur with logits, and so BCE with logits is the safest for practical application and is the loss function I turned to.**

In [ ]:
import math
from torch import nn



a = 0.25 #socially accepted value

p_t = 0.7 #pretending the model guessed with 70% certainty

g = 2 #socially accepted value

focal_seventy = math.pow((1-p_t),g)*math.log(p_t)*-1*a

bce_thirty = -1*(math.log(0.3)+(1-1)*math.log(0.7))

a = 0.25 #socially accepted value

p_t = 0.3 #pretending the model guessed with 30% certainty

g = 2 #socially accepted value

focal_thirty = math.pow((1-p_t), g)*math.log(p_t)*-1*a

bce_seventy = -1*(math.log(0.7)+(1-1)*math.log(0.3))


print(f"When it guessed with 70%% certainty it got %.3f loss. When it guessed with 30%% certainty it got %.3f loss\nwhich is around {Guessed_Minority/Guessed_Majority:.0f} times more! With BCE loss getting {bce_seventy:.3f} and {bce_thirty:.3f} respectively." %
       (focal_seventy, focal_thirty))

When it guessed with 70% certainty it got 0.008 loss. When it guessed with 30% certainty it got 0.147 loss
which is around 13 times more! With BCE loss getting 0.357 and 1.204 respectively.


In [ ]:
predictions = torch.Tensor([[0.7,0.3], [0.4, 0.6]]) # seventy percent certain the first time, thirty percent certain the second
print(predictions)
solutions = [1, 0] # it was detected both times
def focalLoss(x, y):
  loss = 0
  a = 0.25
  g = 2
  for i in range(len(y)):
    if y[i] == 0:
      p_t = x[i][0]
    if y[i] == 1:
      p_t = x[i][1]
    loss += torch.pow((1-p_t), g)*torch.log(p_t)*-1*a
  return loss

print(f'{focalLoss(predictions, solutions):.3f}')

tensor([[0.7000, 0.3000],
        [0.4000, 0.6000]])
0.230


#The Plan

First, we need to determine the best model for this task. This is an image classification model and is based in a computer vision problem over **one** subject, so the best model is a CNN. Originally I had been looking into R-CNNs, as they had piqued my interest while I was preparing the data, but I have decided against them and instead for CNNs.

##Basics of CNNs
CNNs work by using a **kernel** which goes across an image and multiplies weights by the intensity values of each color dimension in an image, generating a feature map, each kernel sliding across the image and finding a part of the image it learned to detect. The pooling layer then takes the output of the CNNs forward pass and reduces the data based on a hyperparameter rule like picking the highest number or using a representative mean for each 2x2 window, which reduces the data by 4x.

##Basics of R-CNNs
R-CNNs, or (Regions with CNN features) uses a _selective search_ algorithm to determine areas of interest by pairing areas of similar intensity, color, or texture. This scales it to around 2,000 regions, or areas of interest. Then every proposed area is put through a CNN with the final classification layer (or head) removed. This output is fed to a Support Vector Machine, which scores the feature vector on the confidence with which it can state that its trained class is or is not appearing in the image. The R-CNN then uses a linear regression model that learns to adjust the areas of interest to fit better around an image, better capturing the identified aspect using NMS (Non-Maximum Suppression), which removes lower confidence scores.

##Why **Not** an R-CNN?
R-CNNs are best for when there are many subjects in an image. This would be useful for this project if the goal were to identify infected bees out of many in a hive, in a high-resolution photograph. However, this project was mainly to determine the health of a **singular** bee, and the best way to do that is a CNN. Concept aside, CNNs are also extremely lightweight, and much more realistic for a starter project.

#The Model

##Specifics
Alright. Hard parts over. Now we have to hardcode the specifics, including the count of convolutional and pooling layers this project needs, the loss function, the optimizer, and other hyperparameters.
To begin by going down the list, the convolutional layer number 1. For this layer, I decided to use valid convolution, meaning that there is no padding. This is because in the images trained on, there are no important details in the corners and so running with padding will increase compute need and time unnecessarily for only a miniscule boost in accuracy, if any at all.
This model is extremely **simple** and was built as a basic example of a convolutional neural network and a method of understanding through practical application.

##How Does the Model Actually Work?
The model runs by sending the data through the first convolutional layer (conv1 in the example below). This layer runs a **kernel**, which is a sliding window performed over an entire image through each channel to find learned patterns in the data (oversimplification). Then, that **feature map** is sent to a pooling layer. In this case it uses a _max_ pool layer, which taxes the maximum value within its own kernel and *summarizes* the data to make it smaller and less computationally heavy. Next it is run through a second convolutional layer to catch larger scale patterns, with the image now being a less detailed version of its previous self. Then it is sent to a fully connected layer. This layer uses a Support Vector Machine, which is just a basic linear regression or classifier model.

In [1]:
import torch
from PIL import Image
from pathlib import Path
import torch.utils.data
import torchvision
from torchvision.transforms import v2
import os
from torch import nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import math

#-------------============== * DATA * ==============-------------#

transforms= v2.Compose([v2.ToImage(), #Converts PIL, tensors and ndarrays into TV Images
                        v2.Resize([160,280]), #resizes to a standard for CNN
                 v2.ToDtype(torch.float32, scale=True), #converts the image to torch.float32 tensor
                 v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]) #normalizes with mean of 0th arg and std of 1st arg

class Custom(torch.utils.data.Dataset):
  def __init__(self, image_dir, transform):
    self.stem_dir = image_dir[:str(image_dir).rfind("/")]
    self.image_dir = Path(image_dir)
    self.transform=transform
    #print(self.image_dir)
    self.images = [
        os.path.join(image_dir, iname) for iname in os.listdir(image_dir) if iname[0] != "."
    ] ####
  def __len__(self):
    return len(self.images) #len
  def __getlabel__(self,id):
    pathToLabel = self.images[id]
    pathToLabel = pathToLabel[pathToLabel.rfind("/")+1:-4]
    pathToLabel = os.path.join(self.stem_dir,"labels", pathToLabel+".txt")
    try:
      with open(pathToLabel) as fileInput:
        label = fileInput.readline().strip()
      if label == "0":
        return 0
      else:
        return 1
    except FileNotFoundError:
      print("File not found: " + str(self.images[id]))
      return 0
  def __getitem__(self, id):
    if self.images[id][self.images[id].rfind("."):] != ".png":
      print("not png " + self.images[id])
      pass
    else:
      try:
        image = Image.open(self.images[id]).convert('RGB') #edit when outside colab because colab adds .ipynb folder
        img = self.transform(image)
        label = int(self.__getlabel__(id))
        return torch.tensor(img), torch.tensor(label, dtype=torch.float32)
      except Exception as e: # Debug step for when images without corresponding labels would appear
        print("Unable to load; corrupt file; {e}")

file_input = input("Test Image Path (example: /content/test/images): ")
testset = Custom(file_input, transform=transforms)
file_input = input("Training Image Path (example: /content/training59/images): ")
customset = Custom(file_input, transform=transforms)
dataloader = DataLoader(customset, batch_size=32, shuffle=True, num_workers=0)
print(customset.__len__())

#-------------============== * LOSS * ==============-------------#

def focalLoss(x, y):
  loss = 0
  g = 2#
  a = 2#



#-------------============== * MODEL * ==============-------------#


#we already have the data tensors with dimensions of (n, c, h, w)
class model(nn.Module):
  def __init__(self, in_channels):
    super().__init__()
    self.in_channels = in_channels
    self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8,
                           kernel_size=3, stride=1)
    self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    self.conv2 = nn.Conv2d(in_channels=8, out_channels=8, kernel_size=3,
                          stride=1)
    self.fc = nn.LazyLinear(out_features=1) #in features can be dynamic
    self.activation = nn.LeakyReLU()
    self.dropout = nn.Dropout(p=0.6) #didn't want to be too aggressive with a 0.5 dropout rate
  def forward(self, photo):
    a = self.conv1(photo)
    a = self.activation(a)
    a = self.pool(a)
    a = self.conv2(a)
    a = self.activation(a)
    a = self.dropout(a)
    a = self.pool(a)
    a = a.reshape(a.shape[0], -1) # flattens data into (b, *) for fully connected layer
    a = self.fc(a)
    return a
mouse = model(3)
optimizer = torch.optim.Adam(mouse.parameters(), lr=0.001)
epochs = int(input("Epochs? "))
loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(2.4)) #2.4 defined in the cited dataset
graph_loss = []
for epoch in range(epochs):
  for batch_images, batch_labels in dataloader:
    optimizer.zero_grad()
    move = mouse.forward(batch_images)
    loss = loss_fn(move.squeeze(dim=1), batch_labels)
    graph_loss.append(loss.item())
    #loss = focalLoss(move.squeeze(dim=1), batch_labels)
    loss.backward()
    optimizer.step()
    probabilities = torch.sigmoid(move)
  print("Epoch " + str(epoch) + " complete.")
  print(loss)

save = input("Save Results? ")
if save.lower() == "y":
  torch.save(mouse.state_dict(), "mouse_weights.pt")
def test():
  print("#-------------============== * TESTING * ==============-------------#")
  dataloader = DataLoader(testset, batch_size=32, shuffle=True, num_workers=0)
  mouse.eval()
  for batch_images, batch_labels in dataloader:
    with torch.inference_mode():
      move = mouse.forward(batch_images)
      print(loss_fn(move, batch_labels.unsqueeze(dim=1)))
test()
save = input("Save Results? ")
if save.lower() == "y":
  torch.save(mouse.state_dict(), "mouse_weights.pt")

KeyboardInterrupt: 

In [ ]:
!rm -rf */.ip.ipynb_checkpoints

##What Went Wrong?
###Class Imbalance

*Written before it was fixed*

  After the model failed to correctly identify a presence of the varroa mite in any of the images, it is likely that a class imbalance, previously discussed, caused the failure. Although a loss function was created to use certainty as a discriminator, some hyperparameters must be adjusted in order to properly generalize the model for the data. In an attempt to find the problem, I eliminated the focal loss by using BCE with logits, which  Another possibility is the use of an oversampling technique, which also would have solved the class imbalance. Under further analysis, the model seemed to only output the value 0, which made it correct a majority of the time, without it identifying correctly any cases of the varroa mite.

*Written after it was fixed*

  The class imbalance was fixed by adjusting the loss function to a more conventional one, BCE with logits. However, out of hope to fix the class imbalance, I added a pos_weight of 2.4, which is the ratio in the cited dataset. This helps increase the change incurred by the misidentification of a minority class as a multiplier by the pos_weight
###Overfitting
*Written before it was fixed*

  In training data the model did exceptionally well, reaching losses of as little as 0.0002, which made me skeptical of overfitting. Upon the use of a test set, a set unseen by the model, it scored extremely high loss, up to roughly 15.4. The next steps are to try drop outs, which are easy to implement and work by occasionally zero-ing out weights, limiting 'memorization' of the training data. If that fails, I will probably try scaling down the model, but I would need to research further on the nuances of scale, especially because my model currently has in features reaching a maximum of 16, already low.

*Written after dropout implemented*

  I implemented a dropout with a p value of 0.4, which is slightly below the norm (0.5), but I didn't want to be too aggressive. The results were significantly better, with loss on the test set at a maximum of around 5. Over the next training run I will run 13 training loops over 0.5 dropout and see whether or not it helps.

##What Can I Learn
With more time over the summer I would like to complete this project with an adequate loss function or proper oversampling technique that fixed the class imbalance. However, I did learn a few things from this model.

- The creation of custom Dataloader classes for the automation of images
- The abilities of a custom loss function, and how focal loss can help reduce class imbalance
- The CNN types, their uses, and why sometimes the original is the best
- The lack of standardization in real data and how to handle inconsistencies
- The concept of failure being a teacher, where I can learn more from the lack of effectiveness of my hand-built model than the success of a copy-pasted template